# 🌳 Composite Design Pattern — Hinglish mein samjho (Beginner Friendly)

Is notebook mein hum **Composite Design Pattern** ko step-by-step, Hinglish comments ke saath samjhenge.
Har code cell ke andar ek **multiline comment (docstring)** hai jisme concept Hinglish mein explain kiya gaya hai.

Pehle dekhenge **bina pattern ke** kya problem aati hai, phir **Composite Pattern** se use solve karenge.
Aakhir mein ek **Practice Question** aur uska **Coding Solution** bhi diya gaya hai taaki tum khud practice kar sako.

In [1]:
"""
COMPOSITE DESIGN PATTERN — KYA HAI?
========================================

Composite ek STRUCTURAL design pattern hai jo objects ko
TREE STRUCTURE (hierarchy) mein organize karta hai, aur CLIENT
ko INDIVIDUAL objects aur GROUPS OF objects, DONO ko SAME
INTERFACE se, UNIFORMLY treat karne deta hai.

SOCHO EK EXAMPLE:
--------------------
Tumhare COMPUTER ke file system mein FILES (jaise "photo.jpg")
aur FOLDERS (jisme AUR files/folders ho sakte hain) hote hain.
Agar tum "display()" ya "delete()" karna chaho, to CHAHE wo EK
FILE ho YA EK PURA FOLDER (jisme 1000 files ho), TUMHARA CODE
SAME REHNA CHAHIYE -- "agar file hai to X karo, agar folder hai
to LOOP chala kar Y karo" jaisi if-else NAHI likhni chahiye.

COMPOSITE isko solve karta hai:
    -> Ek COMMON INTERFACE banao (jaise "display()")
    -> FILES (individual objects) is interface ko IMPLEMENT
       karte hain
    -> FOLDERS (groups) BHI ISI interface ko IMPLEMENT karte
       hain -- lekin ANDAR ANDAR apne CHILDREN (files/folders)
       par WAHI operation RECURSIVELY CALL kar dete hain

PART-WHOLE (WHOLE-PART) HIERARCHY:
---------------------------------------
Ye pattern "PART-WHOLE" relationships represent karta hai --
SIMPLE parts se COMPLEX wholes bante hain. Jaise ek Graphic
Design app mein:
    - PARTS   -> individual shapes (circle, rectangle)
    - WHOLES   -> complex shapes (smiley face = circles + rectangle)
Composite Pattern se HAR DONO ko SAME operations (jaise "draw()")
se handle kar sakte hain.

MAIN COMPONENTS:
--------------------
1. Component   -> COMMON interface jo Leaf aur Composite DONO
                 IMPLEMENT karte hain (e.g. FileSystemComponent)
2. Leaf          -> INDIVIDUAL object, CHILDREN NAHI hote (e.g. File)
3. Composite       -> CONTAINER object jo Leaf YA Composite,
                 DONO ko CHILDREN ki tarah HOLD kar sakta hai
                 (e.g. Directory) -- add/remove/access ke methods
                 deta hai
4. Client            -> Component interface se kaam karta hai,
                 use FARAK NAHI PADTA ki wo Leaf hai ya Composite

KAISE KAAM KARTA HAI:
--------------------------
1. Client, Component interface ke through objects se INTERACT karta hai
2. Leaf objects ACTUAL kaam karte hain
3. Composite objects apne CHILDREN ko STORE/MANAGE karte hain
4. Jab Composite par operation CALL ho, wo apne HAR CHILD par
   USE RECURSIVELY FORWARD kar deta hai

REAL LIFE EXAMPLES:
-----------------------
- File Systems: Files + Folders
- Graphics/GUI Libraries: Shapes + Groups of shapes
- Organization Structures: Employees + Teams/Departments

FAYDE (ADVANTAGES):
------------------------
- HIERARCHICAL structures TREE ki tarah represent hote hain --
  individual aur composite UNIFORMLY treat hote hain
- CLIENT CODE SIMPLE rehta hai -- "leaf hai ya composite" ye
  CHECK NAHI karna padta
- FLEXIBILITY -- naye objects add/remove karna EASY hai
- CODE REUSABILITY -- SAME operations parts aur wholes DONO par
  kaam karte hain

NUKSAAN (DISADVANTAGES):
----------------------------
- IMPLEMENTATION thodi COMPLEX ho sakti hai (common interface
  design karna padta hai)
- DEEP hierarchies mein TRAVERSE karna SLOW ho sakta hai
  (performance overhead)
- COMMON interface ki wajah se INVALID operations bhi POSSIBLE
  ho sakte hain (type safety kam ho jaati hai)
- CHILD REFERENCES store karne se EXTRA MEMORY use hoti hai
"""
print("Concept samajh liya! Ab neeche code mein implement karke dekhte hain.")


Concept samajh liya! Ab neeche code mein implement karke dekhte hain.


## ❌ Pehle dekho: BINA Composite Pattern ke (Problem Code)

Chalo pehle ye dekhte hain ki Composite Pattern KE BINA log File System (Files + Directories)
ko represent karte hain, aur usme kya dikkat aati hai.

In [2]:
"""
BINA COMPOSITE KE — PROBLEM DEMO
--------------------------------------
Yahan File aur Directory DO ALAG, UNRELATED classes hain -- koi
COMMON INTERFACE nahi hai. Client ko HAR BAAR "type check"
(isinstance) karna padta hai ki ye FILE hai ya DIRECTORY, aur
uske hisaab se ALAG LOGIC likhni padti hai.
"""

class FileBad:
    def __init__(self, name, size):
        self.name = name
        self.size = size

    def show_file(self):
        print(f"File: {self.name} ({self.size} bytes)")


class DirectoryBad:
    def __init__(self, name):
        self.name = name
        self.items = []   # ismein File OR Directory dono ja sakte hain, lekin ALAG TAREEKE se

    def add_item(self, item):
        self.items.append(item)

    def show_directory(self):
        print(f"Directory: {self.name}")
        for item in self.items:
            # PROBLEM: Client (yahan Directory khud) ko CHECK karna pad raha hai
            # ki item FILE hai ya DIRECTORY -- aur ALAG METHOD call karna pad raha hai!
            if isinstance(item, FileBad):
                item.show_file()
            elif isinstance(item, DirectoryBad):
                item.show_directory()
            else:
                print("⚠️ Unknown item type!")


# Client code
file1 = FileBad("document.txt", 1024)
file2 = FileBad("image.jpg", 2048)

sub_folder = DirectoryBad("Photos")
sub_folder.add_item(FileBad("vacation.png", 3000))

my_documents = DirectoryBad("My Documents")
my_documents.add_item(file1)
my_documents.add_item(file2)
my_documents.add_item(sub_folder)   # ek Directory ke ANDAR DOOSRI Directory

my_documents.show_directory()


Directory: My Documents
File: document.txt (1024 bytes)
File: image.jpg (2048 bytes)
Directory: Photos
File: vacation.png (3000 bytes)


In [3]:
"""
ISME PROBLEM KYA THI?
-------------------------
1. NO COMMON INTERFACE: FileBad aur DirectoryBad DO BILKUL
   ALAG, UNRELATED classes hain -- inke methods ke NAAM BHI ALAG
   hain ("show_file()" vs "show_directory()"). Client (ya khud
   Directory) ko YE PATA hona chahiye ki KIS OBJECT par KAUNSA
   METHOD call karna hai.

2. TYPE-CHECKING EVERYWHERE: "isinstance(item, FileBad)" jaisi
   if-elif CHAINS HAR JAGAH likhni padti hain jahan bhi hum
   items ko PROCESS karte hain. Agar kal EK NAYA TYPE add ho
   (jaise "Shortcut"), to YE if-elif CHAIN HAR JAGAH DHOONDH KE
   UPDATE karni padegi.

3. NOT SCALABLE: Jaise jaise HIERARCHY DEEP hoti jaayegi (folder
   ke andar folder ke andar folder...), ye isinstance CHECKS aur
   BHI COMPLICATED ho jaayenge.

4. FRAGILE CODE: Agar "else" case (Unknown item type) mein koi
   naya OBJECT TYPE aa jaaye jo humne SOCHA HI NAHI tha, to
   SILENTLY FAIL ho sakta hai ya CRASH ho sakta hai.

YE SAB PROBLEMS "Composite Design Pattern" SOLVE KARTA HAI!
Neeche BEST APPROACH dekhte hain -- File aur Directory, DONO EK
COMMON INTERFACE follow karenge, taaki UNHE UNIFORMLY treat kiya
ja sake.
"""
print("Chalo ab BEST APPROACH (Composite Pattern) dekhte hain...")


Chalo ab BEST APPROACH (Composite Pattern) dekhte hain...


## ✅ Ab dekho: BEST APPROACH — Composite Pattern se solve karte hain

### 1️⃣ Component — `FileSystemComponent`

In [4]:
"""
COMPONENT (FileSystemComponent)
--------------------------------------
Ye COMMON INTERFACE hai jise File (Leaf) aur Directory (Composite)
DONO follow karenge. Isme SIRF EK method hai: display().
"""

from abc import ABC, abstractmethod

class FileSystemComponent(ABC):
    @abstractmethod
    def display(self):
        pass


### 2️⃣ Leaf — `File`

In [5]:
"""
LEAF (File)
---------------
File ek INDIVIDUAL object hai -- iske KOI CHILDREN NAHI hote.
Ye FileSystemComponent interface IMPLEMENT karta hai, aur apna
"display()" method deta hai.
"""

class File(FileSystemComponent):
    def __init__(self, name, size):
        self.name = name
        self.size = size

    def display(self):
        print(f"File: {self.name} ({self.size} bytes)")


### 3️⃣ Composite — `Directory`

In [6]:
"""
COMPOSITE (Directory)
--------------------------
Directory bhi FileSystemComponent interface IMPLEMENT karta hai
-- isliye ye BHI ek "FileSystemComponent" JAISA DIKHTA hai. Lekin
ANDAR SE ye APNE CHILDREN (jo File YA DOOSRI Directory ho sakte
hain) ki EK LIST rakhta hai.

Jab "display()" call ho, ye APNA NAAM print karta hai, aur PHIR
HAR CHILD par "display()" RECURSIVELY call kar deta hai -- CHILD
FILE ho ya DIRECTORY, FARAK NAHI PADTA!
"""

class Directory(FileSystemComponent):
    def __init__(self, name):
        self.name = name
        self.components = []   # File YA Directory, DONO yahan store ho sakte hain

    def add_component(self, component: FileSystemComponent):
        self.components.append(component)

    def display(self):
        print(f"Directory: {self.name}")
        for component in self.components:
            component.display()   # RECURSIVE call -- Leaf ho ya Composite, SAME method!


### 4️⃣ Client — File aur Directory, DONO ko UNIFORMLY treat karta hai

In [7]:
"""
CLIENT CODE
---------------
Ab dekho -- kahin bhi "isinstance()" check NAHI karna pada!
Client SIRF "display()" call karta hai, chahe object File ho ya
Directory.
"""

# Leaf objects (files)
file1 = File("document.txt", 1024)
file2 = File("image.jpg", 2048)
vacation_photo = File("vacation.png", 3000)

# Composite objects (directories) -- NESTED bhi ho sakte hain
photos_folder = Directory("Photos")
photos_folder.add_component(vacation_photo)

my_documents = Directory("My Documents")
my_documents.add_component(file1)
my_documents.add_component(file2)
my_documents.add_component(photos_folder)   # Directory ke ANDAR DOOSRI Directory!

# Client SIRF "display()" call kar raha hai -- File ho ya Directory, SAME call!
my_documents.display()

print("\n✅ Composite Pattern se File aur Directory, DONO ko SAME '.display()' call se handle kiya -- KOI isinstance() check NAHI lagi.")


Directory: My Documents
File: document.txt (1024 bytes)
File: image.jpg (2048 bytes)
Directory: Photos
File: vacation.png (3000 bytes)

✅ Composite Pattern se File aur Directory, DONO ko SAME '.display()' call se handle kiya -- KOI isinstance() check NAHI lagi.


## 📝 Practice Question (Tumhari baari!)

**Problem Statement:**

Ek **Organization Structure System** banao using Composite Design Pattern (jaise document ke
"Real Life Example" mein "Organization Structures" mention hua tha).

Ek Company mein `Employee` (individual) aur `Team` (jisme MULTIPLE employees ya SUB-TEAMS ho
sakte hain) hote hain. Tumhe `show_details()` (jo naam aur salary print kare) ko UNIFORMLY
call karna hai, chahe wo EK employee ho ya PURI team.

Tumhe ye banana hai (File/Directory example jaisi hi structure use karo):

1. `OrgComponent` (Component interface) — abstract methods `show_details()` aur `get_total_salary()`
2. `Employee` (Leaf) — `name`, `salary` attributes; `show_details()` print kare naam+salary; `get_total_salary()` apni salary return kare
3. `Team` (Composite) — `name` attribute, `members` list; `add_member(component)` method; `show_details()` apna naam print kare phir HAR member ka `show_details()` call kare; `get_total_salary()` SAARE members ki salary ka SUM return kare (recursively, sub-teams ke liye bhi)
4. Client code likho jo:
   - 3 `Employee` banaye
   - Ek `Team` ("Backend Team") banaye jisme 2 employees hon
   - Ek BADI `Team` ("Engineering Dept") banaye jisme "Backend Team" (sub-team) aur 1 aur employee ho
   - `show_details()` aur `get_total_salary()` "Engineering Dept" par call kare

👉 Pehle khud try karo neeche wale khaali code cell mein, phir uske baad wale cell mein solution dekho!

In [8]:
"""
YAHAN KHUD LIKHO (Practice Cell)
-----------------------------------
Hint: FileSystemComponent/File/Directory example ko copy karke
sirf naam badal do:
FileSystemComponent -> OrgComponent, File -> Employee, Directory -> Team
display() -> show_details() (aur ek NAYA method get_total_salary() bhi add karna hai)

Apna code neeche likho:
"""

# TODO: class OrgComponent (ABC) banao with abstract show_details() aur get_total_salary()
# TODO: class Employee (Leaf) banao with name, salary
# TODO: class Team (Composite) banao with name, members list, add_member()
# TODO: Client code likho: employees banao, Backend Team banao, Engineering Dept banao (NESTED), test karo


'\nYAHAN KHUD LIKHO (Practice Cell)\n-----------------------------------\nHint: FileSystemComponent/File/Directory example ko copy karke\nsirf naam badal do:\nFileSystemComponent -> OrgComponent, File -> Employee, Directory -> Team\ndisplay() -> show_details() (aur ek NAYA method get_total_salary() bhi add karna hai)\n\nApna code neeche likho:\n'

## ✅ Solution — Practice Question ka Coding Solution

In [9]:
"""
SOLUTION EXPLANATION
-----------------------
Yahan bilkul FileSystemComponent/File/Directory jaisi hi
structure use ki hai, bas ab "Org Structure" bana rahe hain
instead of "File System".

1. OrgComponent    -> Component Interface
2. Employee          -> Leaf
3. Team                -> Composite
4. Neeche client code hai jo NESTED teams banata hai
"""

from abc import ABC, abstractmethod

# ---- 1. Component ----
class OrgComponent(ABC):
    @abstractmethod
    def show_details(self):
        pass

    @abstractmethod
    def get_total_salary(self):
        pass


# ---- 2. Leaf ----
class Employee(OrgComponent):
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def show_details(self):
        print(f"👤 {self.name} -- ₹{self.salary}")

    def get_total_salary(self):
        return self.salary


# ---- 3. Composite ----
class Team(OrgComponent):
    def __init__(self, name):
        self.name = name
        self.members = []   # Employee YA Team, DONO store ho sakte hain

    def add_member(self, component: OrgComponent):
        self.members.append(component)

    def show_details(self):
        print(f"👥 Team: {self.name}")
        for member in self.members:
            member.show_details()

    def get_total_salary(self):
        return sum(member.get_total_salary() for member in self.members)


# ---- 4. Client Code ----
alice = Employee("Alice", 80000)
bob = Employee("Bob", 75000)
charlie = Employee("Charlie", 90000)

backend_team = Team("Backend Team")
backend_team.add_member(alice)
backend_team.add_member(bob)

engineering_dept = Team("Engineering Dept")
engineering_dept.add_member(backend_team)   # NESTED -- team ke andar team!
engineering_dept.add_member(charlie)

engineering_dept.show_details()
print(f"\n💰 Total Salary (Engineering Dept): ₹{engineering_dept.get_total_salary()}")


👥 Team: Engineering Dept
👥 Team: Backend Team
👤 Alice -- ₹80000
👤 Bob -- ₹75000
👤 Charlie -- ₹90000

💰 Total Salary (Engineering Dept): ₹245000


## 🎯 Bonus Challenge

Ab try karo:
1. Ek AUR sub-team `Frontend Team` add karo `Engineering Dept` mein — 2 naye employees ke saath.
   `get_total_salary()` phir se call karo -- kya ye automatically SAHI TOTAL deta hai bina
   `get_total_salary()` mein KOI CHANGE kiye? (Ye Composite ki RECURSIVE power hai!)
2. Ek `count_employees()` method add karo `OrgComponent` mein — jo COUNT kare ki ek `Team`
   (nested sub-teams samet) mein TOTAL KITNE employees hain.
3. Socho: agar `Employee` aur `Team` SAME `OrgComponent` interface follow NA karte (jaise
   humne "BAD" version mein dekha tha), to `get_total_salary()` jaisa RECURSIVE calculation
   likhna KITNA MUSHKIL hota? Comment likh ke explain karo.